In [1]:
import pandas as pd
import json
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler,StandardScaler

current_dir = Path.cwd()
project_root = current_dir.parents[2]



with open(project_root/"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/Domain_data.json", "r") as archivo:
    full_domain = json.load(archivo)

with open(project_root/"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS_Domain_data.json", "r") as archivo:
    updrs_domain = json.load(archivo)

X_multiples= { 'X_STATS':project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS.csv', 
                    'X_V06+STATS': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+STATS.csv',
                      'X_V06+DELTA': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+Deltas.csv'}

y_multiples = { 'target_HY3': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3.csv',
                'target_HY4': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4.csv',
                'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43.csv',
                'target_MCID': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID.csv'}

csv_output_paths = {'UPDRS':{'FULL_SET':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY43/FULL_SET/'},

                                'FEATURE_SELECTION':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/UPDRS/HY43/FEATURE_SELECTION/'}},

                    'FULL':{'FULL_SET':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY43/FULL_SET/'},
                                
                                'FEATURE_SELECTION':{'target_HY43': project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/MODEL_PERFORMANCE_RESULTS/CLASSIFICATION_Multiple/FULL/HY43/FEATURE_SELECTION/'}}}

dominios_full = {
    'X_STATS': {
        'full': full_domain['SC_data'] + full_domain['M_data_STATS'] + full_domain['NM_data_STATS'],
        'motor': full_domain['M_data_STATS'],
        'non_motor': full_domain['NM_data_STATS']
    },

    'X_V06+STATS': {
        'full': full_domain['SC_data']
                + full_domain['M_data_V06'] + full_domain['M_data_STATS']
                + full_domain['NM_data_V06'] + full_domain['NM_data_STATS'],
        'motor': full_domain['M_data_V06'] + full_domain['M_data_STATS'],
        'non_motor': full_domain['NM_data_V06'] + full_domain['NM_data_STATS']
    },

    'X_V06+DELTA': {
        'full': full_domain['SC_data']
                + full_domain['M_data_V06'] + full_domain['M_data_DELTA']
                + full_domain['NM_data_V06'] + full_domain['NM_data_DELTA'],
        'motor': full_domain['M_data_V06'] + full_domain['M_data_DELTA'],
        'non_motor': full_domain['NM_data_V06'] + full_domain['NM_data_DELTA']
    }
}

dominios_updrs = {
    'X_STATS': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS']
                + updrs_domain['UPDRS_NO_MOTOR_STATS']
                + updrs_domain['UPDRS_MOTOR_STATS'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06+STATS': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS']
                + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
                + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06+DELTA': {
        'full': updrs_domain['SC_data']
                + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_delta']
                + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_delta']
                + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_delta'],
        'examen_motor': updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_delta'],
        'impacto_motor': updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_delta'],
        'non_motor': updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_delta']
    }
}



classification_models = {

    "decision_tree": DecisionTreeClassifier(random_state=42,class_weight="balanced"),

    "random_forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "extra_trees": ExtraTreesClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        tree_method="hist",   
        device="cuda",       
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ),


    "adaboost": AdaBoostClassifier(
        algorithm="SAMME", 
        random_state=42
    ),

    "svm": SVC(
        kernel="rbf",
        probability=True,  
        random_state=42
    ),

    "logistic_regression": LogisticRegression(
        random_state=42,
        n_jobs=-1,
        max_iter=10000,
        class_weight="balanced"
    ),

    "knn": KNeighborsClassifier(
        n_jobs=-1
    ),

    "gaussian_nb": GaussianNB()
}

# MODELO ORDINAL

# Modelo de Clasificación Ordinal para Parkinson

Problema

Se dispone de una variable objetivo con tres clases ordenadas:

- **0**: Prodrómico / sano  
- **1**: Parkinson leve (HY 1)  
- **2**: Parkinson moderado-severo (HY ≥ 2)  

Estas clases siguen una estructura ordinal:

0 < 1 < 2

---

Limitación del enfoque multiclass

Los modelos de clasificación tradicionales (multiclass) tratan las clases como independientes, sin considerar su orden.

Esto implica que:

- Error 0 → 1 (leve)  
- Error 0 → 2 (grave)  

son penalizados de la misma forma, lo cual no es adecuado en este contexto clínico.

---

Enfoque de clasificación ordinal

Para incorporar la relación de orden entre clases, se transforma el problema en varias tareas binarias acumulativas.

Cada clase se codifica de la siguiente manera:

- Clase 0 → [ 0, 0]  
- Clase 1 → [ 1, 0]  
- Clase 2 → [ 1, 1]  

Esto equivale a responder a las siguientes preguntas:

2. ¿La clase es ≥ 1?  
3. ¿La clase es ≥ 2?  

---

Entrenamiento del modelo

Se entrena un modelo con múltiples salidas binarias (una por cada nivel).

Ejemplo de modelos utilizados:

- Random Forest  
- Logistic Regression  
- XGBoost  

Cada salida aprende a separar niveles consecutivos de la enfermedad.

---

Predicción

El modelo devuelve probabilidades para cada nivel, por ejemplo:

[0.9, 0.8, 0.2]

Aplicando un umbral (0.5):

[1, 1, 0]

La clase final se obtiene contando los valores positivos consecutivos:

→ Clase 1

---

Ventajas

- Incorpora la **estructura ordinal** de la variable objetivo  
- Penaliza más los errores entre clases lejanas  
- Mejora la coherencia de las predicciones  
- Evita errores en cascada presentes en modelos jerárquicos  

---

Comparación de enfoques

| Método        | Características                          |
|--------------|------------------------------------------|
| Multiclass   | No considera el orden                    |
| Jerárquico   | Divide el problema en etapas             |
| Ordinal      | Modela directamente el orden entre clases|

---

Conclusión

El enfoque ordinal es especialmente adecuado para modelar la progresión de la enfermedad de Parkinson, ya que respeta la naturaleza ordenada de los niveles de severidad y permite obtener predicciones más consistentes y clínicamente interpretables.